In [1]:
import json
import pandas as pd
import os
import numpy as np
from tqdm import tqdm
os.chdir('..')

In [3]:
path = 'utils_notebooks/gas_counterfact.csv'
df_counterfact = pd.read_csv(path)
df_counterfact

,index,original_pair,corrupted_pair,num_of_triplets,notes,difficult_words,valid,token length
0,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,baju anakku cukup cepat . [A] [O] [S] [A] baju...,1,NaN,NaN,True,NaN
1,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,baik sangat lilin . [A] [O] [S] [A] lilin [O] ...,1,NaN,NaN,True,NaN
2,80,tv nya saja yang tidak bagus karena siaran tvn...,nasi saja yang baik sekali karena polanya kere...,1,NaN,NaN,True,NaN
3,134,pelayanannya ramah . [A] [O] [S] [A] pelayanan...,kucing hitam mahal . [A] [O] [S] [A] kucing hi...,1,NaN,NaN,True,NaN
4,136,tempat parkir mobil yang terbatas . [A] [O] [S...,bumbu rendang yang amat nyaman . [A] [O] [S] [...,1,NaN,NaN,True,NaN
...,...,...,...,...,...,...,...,...
70,2315,dpet kamar yang kurang menarik . semoga next d...,dpet kucing yang mengenyangkan sekali . semoga...,1,NaN,NaN,True,NaN
71,2391,kamarnya benar benar sesuai dengan yang ada di...,pianonya sangat pahit kotor tidak berbau enak ...,1,NaN,NaN,True,NaN
72,2393,"tempat transit , makanannya enak . [A] [O] [S]...","tempat transit , lukisannya bau . [A] [O] [S] ...",1,NaN,NaN,True,NaN
73,2409,tidak ada lift . kesulitannya hanya mengangkut...,NaN,1,NaN,NaN,True,NaN


In [4]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

def convert_to_gas_format(data_list):
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the GAS paper.

	Args:
		data_list: A list of dictionaries, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).

	Returns:
		A single string formatted as '(A, O, S); (A, O, S); ...'
	"""
	# A list comprehension to create a formatted string for each dictionary
	# The order of elements in the tuple is specified as A, O, S
	triplets = [f"({item['A']}, {item['O']}, {item['S']})" for item in data_list]
	
	# Join the list of strings together with a semicolon and space
	return "; ".join(triplets)

In [11]:
df_counterfact['original_pair_input'] = df_counterfact['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip())
df_counterfact['original_pair_output'] = df_counterfact['original_pair'].apply(lambda x: convert_to_gas_format(parse_absa_string(x.split('[A] [O] [S]')[-1].strip())))
df_counterfact['original_pair_gas'] = df_counterfact.apply(lambda row: f"{row['original_pair_input']} => {row['original_pair_output']}", axis=1)

df_counterfact['corrupted_pair_input'] = df_counterfact['corrupted_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip() if pd.notna(x) else x)
df_counterfact['corrupted_pair_output'] = df_counterfact['corrupted_pair'].apply(lambda x: convert_to_gas_format(parse_absa_string(x.split('[A] [O] [S]')[-1].strip())) if pd.notna(x) else x)
df_counterfact['corrupted_pair_gas'] = df_counterfact.apply(lambda row: f"{row['corrupted_pair_input']} => {row['corrupted_pair_output']}" if pd.notna(row['corrupted_pair_input']) and pd.notna(row['corrupted_pair_output']) else None, axis=1)

In [12]:
df_counterfact[['index', 'original_pair_gas', 'corrupted_pair_gas']].rename({'original_pair_gas': 'original_pair', 'corrupted_pair_gas': 'corrupted_pair'}).to_csv('utils_notebooks/gas_counterfact_converted.csv', index=False)